In [1]:
# !pip install yfinance

In [2]:
import yfinance as yf
import pandas as pd
import numpy as np

In [3]:
# SK하이닉스의 데이터를 수집
hynix = yf.Ticker("000660.KS")
# 최근 2년간의 데이터를 추출
df = hynix.history(period = '2y')
df.head()

,Open,High,Low,Close,Volume,Dividends,Stock Splits
Date,,,,,,,
2024-05-20 00:00:00+09:00,189403.282097,190289.724915,185759.017178,187236.421875,3592128,0.0,0.0
2024-05-21 00:00:00+09:00,190486.739372,190585.233032,188319.878841,189107.828125,2468109,0.0,0.0
2024-05-22 00:00:00+09:00,189009.336384,194721.968750,188024.399769,194721.968750,3949439,0.0,0.0
2024-05-23 00:00:00+09:00,200434.590469,200927.058750,195214.426688,196987.312500,4681849,0.0,0.0
2024-05-24 00:00:00+09:00,196790.330155,199449.658941,194524.976004,195608.406250,3793872,0.0,0.0


In [4]:
# 종가, 거래량 데이터만 사용
df = df[ ['Close', 'Volume']]

In [5]:
# 기술적 지표 : 20일간의 이동평균선
# 20개의 데이터를 묶어준다. -> rolling(n)
df['MA20'] = df['Close'].rolling(20).mean()
df.iloc[15 : 25, ]

,Close,Volume,MA20
Date,,,
2024-06-11 00:00:00+09:00,209299.031250,3070163,NaN
2024-06-12 00:00:00+09:00,211761.343750,2151331,NaN
2024-06-13 00:00:00+09:00,218655.906250,5777279,NaN
2024-06-14 00:00:00+09:00,217670.968750,3311223,NaN
2024-06-17 00:00:00+09:00,219640.843750,2198340,199942.116406
2024-06-18 00:00:00+09:00,230967.625000,3136231,202128.676563
2024-06-19 00:00:00+09:00,229982.687500,3758652,204172.419531
2024-06-20 00:00:00+09:00,233922.406250,2927358,206132.441406
2024-06-21 00:00:00+09:00,230475.140625,3848394,207806.832813


In [6]:
# 타겟 변수 -> 내일의 종가가 내일의 20일 평균선보다 높은가? (1: 상승돌파, 0 : 하락)
# 인덱스를 한칸씩 위로 올린다. shift(1) -> 1칸씩 내린다, shift(-1) -> -1칸씩 이동(위로 1칸 이동)
df['target'] = (df['Close'].shift(-1) > df['MA20'].shift(-1)).astype(int)

- astype() -> 타입을 변경하는 함수 (문자형 데이터를 숫자형으로 변경할때는 int)
    - 컬럼 내의 데이터가 '1' -> 1
    - '-' -> error
- to_numeric() -> 범주형 데이터를 수치형으로 변경 (문자로 되어있는 숫자를 숫자 타입으로 변경)
    - '1' -> 1
    - '-' -> NaN

In [7]:
df['target'].value_counts()

target
1    296
0    190
Name: count, dtype: int64

In [8]:
# NLP 감성 점수 데이터를 추가
# 원래는 기사를 크롤링하여 감성 점수를 예측하고 입력해야하지만
# 무작위한 데이터를 입력
df['NLP_Sentiment'] = np.random.uniform(-1, 1, len(df))

In [9]:
df['NLP_Sentiment'].describe()

count    486.000000
mean       0.013658
std        0.581036
min       -0.993250
25%       -0.482583
50%        0.004846
75%        0.517137
max        0.994660
Name: NLP_Sentiment, dtype: float64

In [10]:
# !pip install OpenDartReader

In [11]:
# 영업이익 데이터를 추가
import OpenDartReader
from datetime import datetime
import os
from dotenv import load_dotenv

In [12]:
# .env 파일에서 환경변수로드
load_dotenv()

True

In [13]:
api_key = os.getenv('api_key')

In [14]:
df.head(1)

,Close,Volume,MA20,target,NLP_Sentiment
Date,,,,,
2024-05-20 00:00:00+09:00,187236.421875,3592128,NaN,0,-0.672188


In [15]:
# 시차 정보를 제거 -> 시차 정보가 없는 시계열 데이터와 시차 정보가 존재하는 시계열 데이터가 결합 x
df.index = df.index.tz_localize(None)
df.head(1)

,Close,Volume,MA20,target,NLP_Sentiment
Date,,,,,
2024-05-20,187236.421875,3592128,NaN,0,-0.672188


In [16]:
# dart에서 데이터를 수집
dart = OpenDartReader(api_key)

In [17]:
# 현재년도 로드 -> 최근 3년간 목록을 생성
current_year = datetime.now().year
years = [current_year -2, current_year -1, current_year]
years

[2024, 2025, 2026]

In [18]:
# 분기 보고서 codes
report_codes = ['11013', '11012', '11014', '11011']
dart_data_list = []

In [ ]:
import time

In [ ]:
for year in years:
    for code in report_codes:
        try:
            report = dart.finstate('000660', year, code)
            if report is not None:
                op_profit = report[(report['fs_div'] == 'CFS') & (report['account_nm'] == '영업이익')]
                if not op_profit.empty:
                    val = int(op_profit['thstrm_amount'].values[0].replace(',', '')
                        )
                    # code 11013 -> 1분기 보고서
                    if code == '11013' : d = f"{year}-05-15"
                    elif code == '11012' : d = f"{year}-08-14"
                    elif code == '11014' : d = f"{year}-11-14"
                    else : d = f"{year+1}-03-31"

                    report_date = pd.to_datetime(d)
                    if report_date <= datetime.now():
                        dart_data_list.append({'Date' : report_date, 'Operation_Profit' : val})
                time.sleep(1)
        except: continue
dart_data_list

{'status': '013', 'message': '조회된 데이타가 없습니다.'}

{'status': '013', 'message': '조회된 데이타가 없습니다.'}

{'status': '013', 'message': '조회된 데이타가 없습니다.'}



[{'Date': Timestamp('2024-05-15 00:00:00'), 'Operation_Profit': 2886029000000},
 {'Date': Timestamp('2024-08-14 00:00:00'), 'Operation_Profit': 5468536000000},
 {'Date': Timestamp('2024-11-14 00:00:00'), 'Operation_Profit': 7029958000000},
 {'Date': Timestamp('2025-03-31 00:00:00'),
  'Operation_Profit': 23467319000000},
 {'Date': Timestamp('2025-05-15 00:00:00'), 'Operation_Profit': 7440504000000},
 {'Date': Timestamp('2025-08-14 00:00:00'), 'Operation_Profit': 9212851000000},
 {'Date': Timestamp('2025-11-14 00:00:00'),
  'Operation_Profit': 11383390000000},
 {'Date': Timestamp('2026-03-31 00:00:00'),
  'Operation_Profit': 47206319000000},
 {'Date': Timestamp('2026-05-15 00:00:00'),
  'Operation_Profit': 37610283000000}]

In [20]:
dart_df = pd.DataFrame(dart_data_list)

In [21]:
df.head()

,Close,Volume,MA20,target,NLP_Sentiment
Date,,,,,
2024-05-20,187236.421875,3592128,NaN,0,-0.672188
2024-05-21,189107.828125,2468109,NaN,0,-0.749167
2024-05-22,194721.968750,3949439,NaN,0,-0.990603
2024-05-23,196987.312500,4681849,NaN,0,0.824386
2024-05-24,195608.406250,3793872,NaN,0,0.442037


In [22]:
# df를 리셋 인덱스를 하고 결합
df2 = df.reset_index()

In [23]:
pd.merge(df2, dart_df, on='Date', how='left').ffill()

,Date,Close,Volume,MA20,target,NLP_Sentiment,Operation_Profit
0,2024-05-20,1.872364e+05,3592128,NaN,0,-0.672188,NaN
1,2024-05-21,1.891078e+05,2468109,NaN,0,-0.749167,NaN
2,2024-05-22,1.947220e+05,3949439,NaN,0,-0.990603,NaN
3,2024-05-23,1.969873e+05,4681849,NaN,0,0.824386,NaN
4,2024-05-24,1.956084e+05,3793872,NaN,0,0.442037,NaN
...,...,...,...,...,...,...,...
481,2026-05-14,1.970000e+06,6040068,1434950.0,1,-0.814588,4.720632e+13
482,2026-05-15,1.819000e+06,7485233,1469100.0,1,-0.213589,3.761028e+13
483,2026-05-18,1.840000e+06,6481608,1503350.0,1,0.458552,3.761028e+13
484,2026-05-19,1.745000e+06,4575855,1534200.0,1,-0.820794,3.761028e+13


In [24]:
df = pd.merge_asof(df, dart_df, left_index = True, right_on = 'Date', direction='backward')

In [25]:
df.head(1)

,Close,Volume,MA20,target,NLP_Sentiment,Date,Operation_Profit
Date,,,,,,,
2024-05-20,187236.421875,3592128,NaN,0,-0.672188,2024-05-15,2886029000000


In [26]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

In [27]:
x = df[ ['Volume', 'MA20', 'NLP_Sentiment', 'Operation_Profit']] 
y = df['target']

In [28]:
# 시계열 데이터 변환 -> 앞의 80%를 Train으로 뒤의 20%는 Test
split_idx = int(len(df)* 0.8)
X_train, X_test = x.iloc[:split_idx], x.iloc[split_idx :]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

In [29]:
model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [30]:
pred = model.predict(X_test)

In [31]:
print(classification_report(y_test, pred))

              precision    recall  f1-score   support

           0       0.17      1.00      0.29        16
           1       1.00      0.04      0.07        82

    accuracy                           0.19        98
   macro avg       0.58      0.52      0.18        98
weighted avg       0.86      0.19      0.11        98



In [32]:
y_train.value_counts()

target
1    214
0    174
Name: count, dtype: int64

In [33]:
# 클래스 가중치 부여
model = RandomForestClassifier(random_state=42, class_weight="balanced_subsample")

model.fit(X_train, y_train)

pred = model.predict(X_test)

print(classification_report(y_test, pred))

              precision    recall  f1-score   support

           0       0.17      1.00      0.29        16
           1       1.00      0.04      0.07        82

    accuracy                           0.19        98
   macro avg       0.58      0.52      0.18        98
weighted avg       0.86      0.19      0.11        98



In [35]:
# 원본 데이터에서 전일 대비 수익율 / 거래량 변화량 (% -> pct_change()
df['Return'] = df['Close'].pct_change()
df['Volume_change'] = df['Volume'].pct_change()
df.head(3)

,Close,Volume,MA20,target,NLP_Sentiment,Date,Operation_Profit,Return,Volume_change
Date,,,,,,,,,
2024-05-20,187236.421875,3592128,NaN,0,-0.672188,2024-05-15,2886029000000,NaN,NaN
2024-05-21,189107.828125,2468109,NaN,0,-0.749167,2024-05-15,2886029000000,0.009995,-0.312912
2024-05-22,194721.968750,3949439,NaN,0,-0.990603,2024-05-15,2886029000000,0.029688,0.600188


In [37]:
# 이동평균선은 절대 수치 대신에 '이격도'로 변환
# 이격도 : 현재 주가가 20일 이평선으로부터 위/아래로 몇 %나 떨어져있는가?
df['Dist_MA20'] = (df['Close'] - df['MA20']) / df['MA20']
df.head()

,Close,Volume,MA20,target,NLP_Sentiment,Date,Operation_Profit,Return,Volume_change,Dist_MA20
Date,,,,,,,,,,
2024-05-20,187236.421875,3592128,NaN,0,-0.672188,2024-05-15,2886029000000,NaN,NaN,NaN
2024-05-21,189107.828125,2468109,NaN,0,-0.749167,2024-05-15,2886029000000,0.009995,-0.312912,NaN
2024-05-22,194721.968750,3949439,NaN,0,-0.990603,2024-05-15,2886029000000,0.029688,0.600188,NaN
2024-05-23,196987.312500,4681849,NaN,0,0.824386,2024-05-15,2886029000000,0.011634,0.185447,NaN
2024-05-24,195608.406250,3793872,NaN,0,0.442037,2024-05-15,2886029000000,-0.007000,-0.189664,NaN


In [38]:
df.dropna(inplace = True)
df.head()

,Close,Volume,MA20,target,NLP_Sentiment,Date,Operation_Profit,Return,Volume_change,Dist_MA20
Date,,,,,,,,,,
2024-06-17,219640.843750,2198340,199942.116406,1,-0.504643,2024-05-15,2886029000000,0.009050,-0.336094,0.098522
2024-06-18,230967.625000,3136231,202128.676563,1,-0.204219,2024-05-15,2886029000000,0.051570,0.426636,0.142676
2024-06-19,229982.687500,3758652,204172.419531,1,0.823022,2024-05-15,2886029000000,-0.004264,0.198461,0.126414
2024-06-20,233922.406250,2927358,206132.441406,1,0.042252,2024-05-15,2886029000000,0.017131,-0.221168,0.134816
2024-06-21,230475.140625,3848394,207806.832813,1,0.811104,2024-05-15,2886029000000,-0.014737,0.314630,0.109084


In [40]:
x = df [ ['NLP_Sentiment', 'Operation_Profit', 'Return', 'Volume_change', 'Dist_MA20']]
y = df['target']

split_idx = int(len(df) * 0.8)
X_train, X_test = x.iloc[: split_idx], x.iloc[split_idx:]
y_train, y_test = y.iloc[: split_idx], y.iloc[split_idx:]

model_final = RandomForestClassifier(random_state=42, class_weight='balanced_subsample')
model_final.fit(X_train, y_train)

pred = model_final.predict(X_test)

print(classification_report(y_test, pred))

              precision    recall  f1-score   support

           0       0.64      0.56      0.60        16
           1       0.91      0.94      0.92        78

    accuracy                           0.87        94
   macro avg       0.78      0.75      0.76        94
weighted avg       0.87      0.87      0.87        94

